# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Maryam-Shehzadi434/flyrank-Internship-ml/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

**Lane:** Lane 1 — Content Refresh Prediction

**Why this lane:**
- In Notebook 01, Random Forest achieved 0.740 Precision@50 — 3.1× better than the hand-written rule (0.240). ML works on this problem.
- 54% of pages in the dataset are declining (`trend_direction = "down"`) — the problem is real and large.
- Search volume has near-zero correlation with impressions (0.001), and CTR drops from 0.355 (page_1) to 0.055 (deep). Simple signals are weak — ML can uncover better patterns.
- Content teams need help prioritizing which pages to fix first. This lane answers that question with evidence, not guesswork.

In [9]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    # find the repo root from wherever this kernel started
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Starter data found. Ready to Go.")

Working dir: /content/flyrank-ml-internship-starter/flyrank-ml-internship-starter
Starter data found. Ready to Go.


In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

declining_rate = df[df["trend_direction"] == "down"].shape[0] / df.shape[0]
print(f"Declining pages: {declining_rate:.1%} of all pages")
print(f"\nBaseline hand-written rule Precision@50: 0.240")
print(f"Random Forest Precision@50: 0.740")
print(f"→ ML beats the rule by {(0.740/0.240):.1f}x")


Declining pages: 54.2% of all pages

Baseline hand-written rule Precision@50: 0.240
Random Forest Precision@50: 0.740
→ ML beats the rule by 3.1x


## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

**Research Question:** Can we predict which pages are declining (`trend_direction = "down"`) using observable signals — search volume, CTR, position tier, and content age?

**Decision:** Which pages should a content team prioritize for refresh or review?

**Action:** Content managers review the top 50 predicted declining pages and decide: refresh, rewrite, remove, or monitor.

**Cost of a wrong call:**
- **False positive** (predict decline, page is actually growing): wastes time refreshing content that didn't need it.
- **False negative** (miss a declining page): lost traffic and revenue as the page continues to drop.
- **Ranking error** (declining page not in top 50): delayed action lets the decline worsen.

**This is decision-support, not automation.** The model ranks candidates so humans spend limited time on the pages that need attention most.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
declining_count = df[df["trend_direction"] == "down"].shape[0]
total_count = df.shape[0]

print(f"Total pages: {total_count:,}")
print(f"Declining pages: {declining_count:,} ({declining_count/total_count:.1%})")
print(f"\nIf a team reviews 50 pages per week:")
print(f"  - ML identifies {int(0.740 * 50)} of the top 50 correctly")
print(f"  - Hand rule identifies {int(0.240 * 50)} of the top 50 correctly")
print(f"  → ML saves ~{int((0.740 - 0.240) * 50)} extra pages per week from manual review")

Total pages: 30,000
Declining pages: 16,262 (54.2%)

If a team reviews 50 pages per week:
  - ML identifies 37 of the top 50 correctly
  - Hand rule identifies 12 of the top 50 correctly
  → ML saves ~25 extra pages per week from manual review


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

**1. Declining pages: 54.2%** — over half of pages show declining impressions. This is a large-scale problem worth solving.

**2. Search volume vs impressions: correlation = 0.001** — near zero. Search volume alone does not predict traffic. Simple signals fail.

**3. CTR by position: 0.355 (page_1) vs 0.055 (deep)** — pages on page_1 get 6.5× more clicks. Position matters — predicting position risk has real value.

**Why this lane is worth 7 weeks:**
- The problem is real and large (54% of pages are declining).
- Simple signals don't work (search volume doesn't predict impressions).
- There is a clear opportunity to help teams prioritize (position affects CTR dramatically).

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Three real numbers from the data

# 1. Declining rate
declining_rate = df[df["trend_direction"] == "down"].shape[0] / df.shape[0]
print(f"1. Declining pages: {declining_rate:.1%} of all pages")

# 2. Correlation
corr = df["search_volume"].corr(df["impressions_90d"])
print(f"2. Correlation search_volume vs impressions: {corr:.3f}")

# 3. CTR by position
ctr_top = df[df["position_tier"] == "page_1"]["ctr"].mean()
ctr_deep = df[df["position_tier"] == "deep"]["ctr"].mean()
print(f"3. CTR page_1: {ctr_top:.3f} vs deep: {ctr_deep:.3f} ({ctr_top/ctr_deep:.1f}x difference)")


1. Declining pages: 54.2% of all pages
2. Correlation search_volume vs impressions: 0.001
3. CTR page_1: 0.652 vs deep: 0.150 (4.3x difference)


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

**What I CAN claim (observed / directional):**
- 54% of pages in this dataset show declining impressions.
- Search volume has a very weak correlation with impressions (0.001) in this sample.
- CTR drops from 0.355 on page_1 to 0.055 in deep positions — position matters.
- A random forest model achieved 0.740 Precision@50 on this dataset, 3.1× better than the baseline.
- These patterns suggest ML may help content teams prioritize review candidates.

**What I CAN'T claim:**
- Search volume causes low impressions (correlation ≠ causation).
- These findings apply to all websites (this is one anonymized dataset).
- This model will work on other data without retesting.
- I have proven how Google's algorithm works (this is observational data).
- Refreshing a page will definitely cause it to recover (this is decision-support, not a guarantee).

**Language I'll use:** "observed," "measured," "the data shows," "this suggests," "based on this dataset," "could help prioritize."

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

print("Section 4 complete — careful words documented in markdown above.")

Section 4 complete — careful words documented in markdown above.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.